# Model D v1 — original checkpoint verification

Run on a Colab CUDA runtime. This workflow loads the **original Model B + epoch-17 HVI head** and never trains. Reported 21.0716 dB validation / 18.8959 dB Test PSNR remain historical until verification succeeds.

The HVI head consumes quantized Model B RGB, exactly as in the original PNG-cache experiment. Results are written separately from the source checkpoints. Run validation before Test; the CLI enforces this requirement.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
from datetime import datetime, timezone
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/myRetinexformer_model_d")
REPO_URL = "https://github.com/Silly-Gooses/myRetinexformer.git"
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "model_d", "--single-branch", REPO_URL, str(REPO)], check=True)
assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
assert subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO, text=True).strip() == "model_d"
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Resolved model_d commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
import torch
assert torch.cuda.is_available(), "Select a CUDA GPU runtime"
print("Python:", sys.version, "Torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))
sys.path.insert(0, str(REPO))


## Original inputs and separate output directory
Only change paths to locate the original files. Do not substitute later Model D checkpoints. A missing head is a blocker, not a reason to retrain.


In [ ]:
DRIVE = Path("/content/drive/MyDrive")
SCREENING = DRIVE / "RetinexFormer_NAF_Screening"
DATA_ROOT = DRIVE / "datasets/LOLv2/Real_captured"
MODEL_B = SCREENING / "model_B_optionA_batch8_patch128_lr1e-5_from_batch8best/low_lr_1e-5_1000steps/best.pth"
HEAD = SCREENING / "model_D_HVI_from_ModelB_OptionA_v1/best_model_d_head.pth"
SPLIT = SCREENING / "model_B_v1_full_data_1000steps/split.json"
RUN = SCREENING / "model_D_v1_verification" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
for path in (MODEL_B, HEAD, SPLIT):
    assert path.is_file(), f"Original dependency unavailable: {path}"
for part in ("Train/Low", "Train/Normal", "Test/Low", "Test/Normal"):
    assert (DATA_ROOT / part).is_dir(), f"Missing dataset directory: {part}"
original_split = json.loads(SPLIT.read_text())
IMAGE = DATA_ROOT / "Train/Low" / original_split["val"][0]
print("Verification output:", RUN)
BASE = [sys.executable, "-m", "model_d.cli"]
COMMON = ["--model-b", str(MODEL_B), "--head", str(HEAD), "--device", "cuda"]
def run_cli(command, *args):
    subprocess.run(BASE + [command] + COMMON + list(args), cwd=REPO, check=True)


## Strict checkpoint and independent notebook tests
The preserved reference fixture is hash-checked. Only model/conversion definitions are evaluated; its training and setup cells are never executed.


In [ ]:
test_env = dict(os.environ, MODEL_D_BACKBONE=str(MODEL_B), MODEL_D_HEAD=str(HEAD), MODEL_D_IMAGE=str(IMAGE))
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_model_d.py", "-v"], cwd=REPO, env=test_env, check=True)
run_cli("verify", "--input", str(IMAGE), "--output-dir", str(RUN / "parity"))


## Validation — frozen 69 images
Validation must pass notebook parity and historical metric tolerances before Test. A mismatch stops the workflow; inspect the report rather than tune on Test.


In [ ]:
run_cli("evaluate", "--data-root", str(DATA_ROOT), "--split-file", str(SPLIT), "--split", "val", "--output-dir", str(RUN / "validation"))
validation = json.loads((RUN / "validation/report.json").read_text())
print(json.dumps(validation["metrics"], indent=2))
print("Checkpoint hashes:", json.dumps(validation["checkpoints"], indent=2))


## Official Test — fixed original pair, no tuning
Run after successful validation. Model B and Model D images, per-image measurements, and provenance are saved together.


In [ ]:
run_cli("evaluate", "--data-root", str(DATA_ROOT), "--split-file", str(SPLIT), "--split", "test", "--validation-report", str(RUN / "validation/report.json"), "--output-dir", str(RUN / "test"), "--save-images")
test = json.loads((RUN / "test/report.json").read_text())
print(json.dumps(test["metrics"], indent=2))
print("Preserve the reports and original head SHA256:", RUN)
